# TRELLIS.2 批次版：Drive 資料夾內所有圖片 → 多個面數的 GLB（Colab）

把 `FOLDER` 底下的每一張圖依序轉成帶紋理的 GLB，**存回同一個資料夾**：

- `<原檔名>.glb`：約 100 萬面的高模（`HIGH_TARGET`、4096 貼圖），給 Blender 烘焙法線 / 顏色用。
- `<原檔名>_<面數>.glb`：`TARGETS` 清單裡每個面數各一份（例如 `snailer_tank_20000.glb`、2048 貼圖）。

**為什麼在 Colab 就降面**：`to_glb` 是「先減面 → 再展開 UV → 再從原始體素資料取樣貼圖」，在這裡降面時網格還沒有 UV 接縫，不會像事後在 Blender 對成品 GLB 降面那樣破碎。

**事前準備**
- 執行階段選 **L4 或 A100**（需 24GB 以上顯存）。
- 左側「🔑 密鑰」新增 `HF_TOKEN`，並打開此筆記本的存取權。
- 把要轉的圖放進 Drive 的 `FOLDER`（只處理該資料夾第一層，不含子資料夾）。

**行為**
- 每個輸出檔獨立判斷：已存在就跳過（可安心重跑、續跑）；要重做就把 `OVERWRITE` 設 `True`。
- 同一張圖只生成一次，再用同一份結果輸出所有面數。
- 單張失敗（例如顯存不足）會記錄後繼續下一張，最後列出摘要。

**重啟規則**：若第 5 格提示 transformers 不是 4.x，請「執行階段 → 重新啟動工作階段」（不要刪除），再從第 4 格（掛載 Drive）往下跑。

跑完後記得「中斷並刪除執行階段」，或把 `AUTO_DISCONNECT` 設為 `True`。

In [ ]:
# ── 設定 ──────────────────────────────────────────
FOLDER = "/content/drive/MyDrive/tmp/tank"   # 圖片所在資料夾；GLB 也存回這裡
OVERWRITE = False                            # False：已有同名輸出就跳過

HIGH_TARGET = 1000000        # 高模面數（輸出 <檔名>.glb）；設 None 就不輸出高模
HIGH_TEXTURE_SIZE = 4096
TARGETS = [10000]   # 低面數清單（輸出 <檔名>_<面數>.glb）；可自行增減，[] 則只輸出高模
LOW_TEXTURE_SIZE = 2048

AUTO_DISCONNECT = False      # True：全部跑完自動釋放執行階段（停止計費）

In [ ]:
%%bash
cd /content
[ -d TRELLIS.2 ] || git clone -b main https://github.com/microsoft/TRELLIS.2.git --recursive
cd TRELLIS.2 || exit 1
# 不裝 --flash-attn：Colab 目前的 PyTorch 2.11 / CUDA 13 / Python 3.13 組合編譯不起來，改走下面的 SDPA 補丁
. ./setup.sh --basic --nvdiffrast --nvdiffrec --cumesh --o-voxel --flexgemm
# RMBG 鏡像的自訂程式碼需要 transformers 4.x（DINOv3 需要 >= 4.56）
pip install -q "transformers>=4.56,<5"

In [ ]:
# 稀疏注意力補丁：把 xformers 分支改成逐序列 torch SDPA（不需要 flash-attn / xformers）
# 必須在 import trellis2 之前執行；可重複執行
import re

path = "/content/TRELLIS.2/trellis2/modules/sparse/attention/full_attn.py"
src = open(path).read()

pattern = re.compile(
    r"    if config\.ATTN == 'xformers':\n.*?(?=    elif config\.ATTN == 'flash_attn':\n)",
    re.S,
)

new_branch = """    if config.ATTN == 'xformers':
        # patched: per-sequence torch SDPA (no flash-attn / xformers CUDA needed)
        if num_all_args == 1:
            q, k, v = qkv.unbind(dim=1)
        elif num_all_args == 2:
            k, v = kv.unbind(dim=1)
        outs = []
        qi = ki = 0
        for lq, lk in zip(q_seqlen, kv_seqlen):
            qs = q[qi:qi + lq].transpose(0, 1).unsqueeze(0)    # [1, H, Lq, Ci]
            ks = k[ki:ki + lk].transpose(0, 1).unsqueeze(0)    # [1, H, Lk, Ci]
            vs = v[ki:ki + lk].transpose(0, 1).unsqueeze(0)    # [1, H, Lk, Co]
            o = torch.nn.functional.scaled_dot_product_attention(qs, ks, vs)
            outs.append(o[0].transpose(0, 1))                   # [Lq, H, Co]
            qi += lq
            ki += lk
        out = torch.cat(outs, dim=0)
"""

assert len(pattern.findall(src)) == 1, "找不到唯一的 xformers 分支，停止"
open(path, "w").write(pattern.sub(lambda m: new_branch, src))
print("patched")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from PIL import Image

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp"}
folder = Path(FOLDER)
assert folder.is_dir(), f"找不到資料夾：{folder}"

# 要輸出的版本：(面數, 貼圖大小, 檔名後綴)
variants = []
if HIGH_TARGET:
    variants.append((HIGH_TARGET, HIGH_TEXTURE_SIZE, ""))
variants += [(t, LOW_TEXTURE_SIZE, f"_{t}") for t in TARGETS]
assert variants, "HIGH_TARGET 與 TARGETS 都是空的"

def out_path(p, suffix):
    return folder / f"{p.stem}{suffix}.glb"

# 同名不同副檔名（a.jpg 與 a.png）會輸出到同一個 a.glb，只取第一張
image_paths, seen = [], set()
for p in sorted(folder.iterdir()):
    if p.suffix.lower() not in IMG_EXT or p.name.startswith("."):
        continue
    if p.stem in seen:
        print(f"略過（同名衝突）: {p.name}")
        continue
    seen.add(p.stem)
    image_paths.append(p)

assert image_paths, f"{folder} 裡沒有圖片（{sorted(IMG_EXT)}）"

def missing_variants(p):
    return [v for v in variants if OVERWRITE or not out_path(p, v[2]).exists()]

n_todo = sum(1 for p in image_paths if missing_variants(p))
print(f"共 {len(image_paths)} 張圖，需要處理 {n_todo} 張；每張輸出 {len(variants)} 個版本：")
for p in image_paths:
    miss = missing_variants(p)
    print("  ", p.name, "→", [out_path(p, v[2]).name for v in miss] if miss else "全部已存在，跳過")

In [ ]:
import os, sys, subprocess
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"   # 模型快取放 Drive，下次不用重下載
os.environ["ATTN_BACKEND"] = "sdpa"
os.chdir("/content/TRELLIS.2")
sys.path.insert(0, "/content/TRELLIS.2")

import transformers
assert transformers.__version__.startswith("4."), (
    f"transformers 是 {transformers.__version__}，需要 4.x：請重新啟動工作階段後從第 4 格往下跑")

# gated 模型改用鏡像（全新 clone 後必須重做）；權重來源非官方，授權仍以原作者條款為準
PAIRS = [
    ("facebook/dinov3-vitl16-pretrain-lvd1689m", "camenduru/dinov3-vitl16-pretrain-lvd1689m"),
    ("briaai/RMBG-2.0", "1038lab/RMBG-2.0"),
]

from huggingface_hub import list_repo_files
for _, new in PAIRS:
    files = list_repo_files(new)
    assert "config.json" in files, f"{new} 缺 config.json，停止"
    assert any(f.endswith((".safetensors", ".bin", ".pth")) for f in files), f"{new} 沒有權重檔，停止"

for old, new in PAIRS:
    subprocess.run(
        f"grep -rl '{old}' /content/TRELLIS.2 /root/.cache/huggingface /content/drive/MyDrive/hf_cache 2>/dev/null"
        f" | xargs -r sed -i 's#{old}#{new}#g'",
        shell=True)

# BiRefNet 沒有 embedding / encoder-decoder 權重需要綁定；跳過以避開 get_text_config 錯誤
from transformers import modeling_utils
modeling_utils.PreTrainedModel.tie_embeddings_and_encoder_decoder = lambda self: None

from trellis2.pipelines import Trellis2ImageTo3DPipeline
import trellis2.modules.attention.config as attn_cfg
import trellis2.modules.sparse.config as sp_cfg
import o_voxel

attn_cfg.BACKEND = "sdpa"
sp_cfg.ATTN = "xformers"   # 實際走補丁後的 torch SDPA 分支
print("attention:", attn_cfg.BACKEND, "| sparse:", sp_cfg.ATTN)

pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.cuda()

In [ ]:
import gc, time, traceback, torch

done, failed, skipped = [], [], []
t_all = time.time()
for i, p in enumerate(image_paths, 1):
    todo = missing_variants(p)
    if not todo:
        skipped.append(p)
        continue

    print(f"[{i}/{len(image_paths)}] {p.name} ...")
    t0 = time.time()
    mesh = glb = None
    try:
        mesh = pipeline.run(Image.open(p))[0]      # 最花時間的一步，只做一次
        mesh.simplify(16777216)                     # nvdiffrast 上限

        # 面數由高到低輸出
        for target, tex, suffix in sorted(todo, key=lambda v: -v[0]):
            out = out_path(p, suffix)
            t1 = time.time()
            try:
                glb = o_voxel.postprocess.to_glb(
                    vertices=mesh.vertices,
                    faces=mesh.faces,
                    attr_volume=mesh.attrs,
                    coords=mesh.coords,
                    attr_layout=mesh.layout,
                    voxel_size=mesh.voxel_size,
                    aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
                    decimation_target=target,
                    texture_size=tex,
                    remesh=True,
                    remesh_band=1,
                    remesh_project=0,
                    verbose=False,
                )
                tmp = out.with_suffix(".glb.part")   # 先寫暫存檔，避免中斷留下壞掉的 GLB
                glb.export(str(tmp), extension_webp=True)
                tmp.replace(out)
                done.append(out)
                print(f"   saved: {out.name}（{target} 面，{out.stat().st_size / 1e6:.1f} MB，{time.time() - t1:.0f}s）")
            except Exception:
                failed.append(out)
                print(f"   失敗: {out.name}")
                traceback.print_exc()
            finally:
                glb = None
                gc.collect()
                torch.cuda.empty_cache()
        print(f"   {p.name} 共 {time.time() - t0:.0f}s")
    except Exception:
        failed.append(p)
        print(f"   生成失敗: {p.name}")
        traceback.print_exc()
    finally:
        mesh = glb = None
        gc.collect()
        torch.cuda.empty_cache()

print(f"\n完成 {len(done)} 個檔案、跳過 {len(skipped)} 張圖、失敗 {len(failed)}，共 {time.time() - t_all:.0f}s")
for f in failed:
    print("失敗：", getattr(f, "name", f))

In [ ]:
if AUTO_DISCONNECT:
    from google.colab import runtime
    runtime.unassign()   # 釋放執行階段，停止消耗運算單元